# Homework 3

This file contains the solutions to the computational problems in Homework 3 from MATH 5750 taught by Braxton Osting during the Fall 2026 semester.

This code is a mix between my own work, reference code released by Braxton Osting, and code generated by ChatGPT.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import kneighbors_graph, NearestNeighbors
from scipy.sparse.csgraph import connected_components
from scipy.linalg import eigh

Create concentric circles

In [4]:
n = 500

X, y_true = make_circles(n_samples=n, factor=0.45, noise=0.05, random_state=0)

Run $k$-means on the concentric circles

In [5]:
kmeans = KMeans(n_clusters=2, n_init=10, random_state=0)
y_kmeans = kmeans.fit_predict(X)

ari_kmeans = adjusted_rand_score(y_true, y_kmeans)

Run spectral clustering on the concentric circles with $k_{nn}=10$ and $\epsilon$ equal to the median distance from a point to its 10th nearest neighbor.

In [9]:
nn = NearestNeighbors(n_neighbors=11)
nn.fit(X)

distances, indices = nn.kneighbors(X)

epsilon = np.median(distances[:, 10])

print("epsilon =", epsilon)

k_nn = 10

A = kneighbors_graph(
    X,
    n_neighbors=k_nn,
    mode="distance",
    include_self=False
)

A = A.maximum(A.T)

W = A.copy()
W.data = np.exp(-(W.data ** 2) / (2 * epsilon ** 2))

epsilon = 0.12010238417011855


Count connected components to answer question in the homework

In [10]:
n_components, component_labels = connected_components(
    W,
    directed=False
)

print("Number of connected components:", n_components)

Number of connected components: 2


Run spectral clustering with the $W$ matrix defined in the problem

In [12]:
spectral = SpectralClustering(n_clusters=2, affinity="precomputed", random_state=0, assign_labels="kmeans"
)

y_spectral = spectral.fit_predict(W)

ari_spectral = adjusted_rand_score(y_true, y_spectral)

print("k-means ARI:", ari_kmeans)
print("spectral clustering ARI:", ari_spectral)

k-means ARI: -0.001863710783495236
spectral clustering ARI: 1.0


c:\Python313\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:325: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Now compute sepctral clustering using $L_{sym} = I - D^{-1/2}WD^{-1/2}$

In [13]:
W_dense = W.toarray()

degrees = W_dense.sum(axis=1)
D_inv_sqrt = np.diag(1 / np.sqrt(degrees))

L_sym = np.eye(n) - D_inv_sqrt @ W_dense @ D_inv_sqrt

eigenvalues, eigenvectors = eigh(L_sym)

phi1 = D_inv_sqrt @ eigenvectors[:, 1]
phi2 = D_inv_sqrt @ eigenvectors[:, 2]